In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import os
import sys

In [2]:
sys.path.append('/Users/aksharadarapaneni/data-analyst-portfolio/project-1-ecommerce')

from data_loader import load_data

orders, order_items, order_payments, order_reviews, products, sellers, customers, category_translation, orders_delivered = load_data()

In [8]:
reviews_with_text = order_reviews[order_reviews['review_comment_message'].notna()]

In [9]:
reviews_with_text.shape

(40977, 7)

In [10]:
reviews_with_text.groupby('review_score').count()

,review_id,order_id,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
review_score,,,,,,
1,8745,8745,1789,8745,8745,8745
2,2145,2145,458,2145,2145,2145
3,3557,3557,737,3557,3557,3557
4,5976,5976,1433,5976,5976,5976
5,20554,20554,5422,20554,20554,20554


In [15]:
negative_reviews=reviews_with_text[(reviews_with_text['review_score']==1)| (reviews_with_text['review_score'] == 2)] 

In [17]:
negative_reviews.shape

(10890, 7)

In [18]:
sample_reviews = negative_reviews['review_comment_message'].sample(100, random_state=42).tolist()
print(sample_reviews[:3])

['n chegou', 'Um dos relogios veio aberto sem tampa e sem funcionar não recomendo a ninguem essa empresa', 'O produto chegou em 2 entregas, só que bem mal embalado, sem aviso de frágil, num papel pardo. Entregaram 2 pacotes em um dia e um pacote no outro. Pacote amassado nas pontas, peças quebradas. Absurdo']


In [27]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv('/Users/aksharadarapaneni/data-analyst-portfolio/project-1-ecommerce/.env')

client = Groq(api_key=os.getenv('GROQ_API_KEY'))
print("Groq connected!")

Groq connected!


In [29]:
reviews_text = '\n'.join(sample_reviews)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": f"""
The following are customer reviews from a Brazilian e-commerce platform (Olist).
The reviews are in Portuguese.

Analyze these reviews and identify the TOP 5 complaint themes.
For each theme provide:
1. Theme name (in English)
2. Brief description
3. Approximate % of reviews mentioning it

Reviews:
{reviews_text}

Respond in English only.
"""
        }
    ]
)

print(response.choices[0].message.content)

Based on the reviews, I have analyzed the data and identified the TOP 5 complaint themes.

**1. Late/Delayed Delivery (25.6%)**
This theme involves customers complaining about the late delivery of their orders, often exceeding the estimated delivery time.

**2. Product Defects/Incorrect Items (18.1%)**
This theme includes complaints about products being received with defects, not matching the description or image, or receiving a different product than what was ordered.

**3. Missing Items (12.5%)**
Customers are complaining about receiving incomplete orders, with some items missing from the package.

**4. Poor Quality (10.3%)**
This theme encompasses complaints about the poor quality of the products received, including flimsy materials, low-quality construction, or not meeting expectations.

**5. Communication/Response Issues (8.5%)**
This theme involves customers expressing frustration with the company's communication and response to complaints, including difficulties in getting in to

In [30]:
ai_themes = response.choices[0].message.content

In [31]:
summary_prompt = f"""
You are a senior business analyst presenting findings to the C-suite of Olist, a Brazilian e-commerce platform.

Write a professional 3-paragraph executive summary based on these analysis findings:

REVENUE:
- Total revenue: R$13,591,643.70
- Top category: Health & Beauty at R$1,233,131.72
- Monthly revenue grew 780% from Jan 2017 to Nov 2017
- Black Friday spike in Nov 2017, followed by 26% decline in Dec 2017

CUSTOMER RETENTION:
- 94% of customers (90,557) never returned after first purchase
- Revenue leakage: R$12,493,089 (91.9% of total revenue) from one-time buyers
- Only 2,573 customers made 2+ purchases

DELIVERY:
- 8.1% of orders arrive late
- Late orders average 2.27 stars vs 4.29 for on-time orders
- Worst states: Amazonas (33.3% late), Maranhão (19.4% late)

CUSTOMER SEGMENTS (RFM):
- Champions: 15,009 customers
- Loyal: 22,424 customers
- At Risk: 18,577 customers
- Lost: 18,639 customers

TOP COMPLAINT THEMES:
- Late delivery: 25.6%
- Product defects: 18.1%
- Missing items: 12.5%

Write 3 paragraphs: 1) Business performance, 2) Critical problems, 3) Recommended actions.
"""

summary_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": summary_prompt}]
)

executive_summary = summary_response.choices[0].message.content
print(executive_summary)

**Executive Summary to the C-Suite of Olist**

**Business Performance**

Our analysis provides a comprehensive overview of Olist's business performance in the past year. Notably, our revenue growth has been exceptional, with a remarkable 780% increase from January to November 2017, driven by strategic initiatives and market opportunities. The Health & Beauty category has emerged as a top performer, yielding R$1,233,131.72. While we experienced a Black Friday spike, followed by a 26% decline in December 2017, our monthly revenue has consistently demonstrated upward momentum. Our total revenue reached R$13,591,643.70 at the end of the analysis period.

**Critical Problems**

Our analysis has revealed several critical areas that require immediate attention to optimize our customer experience and improve operational efficiency. First, customer retention remains a significant challenge, with 94% of customers never returning after their first purchase. This has resulted in substantial revenu

In [32]:
with open('../README.md', 'w') as f:
    f.write("# Customer Experience & Revenue Intelligence Report\n\n")
    f.write("## Executive Summary\n\n")
    f.write(executive_summary)
    f.write("\n\n## Key Findings\n\n")
    f.write("- 94% of customers never returned after first purchase\n")
    f.write("- R$12.5M revenue leakage (91.9% of total revenue) from one-time buyers\n")
    f.write("- Late delivery scores 2.27 stars vs 4.29 for on-time orders\n")
    f.write("- Amazonas (33.3%) and Maranhão (19.4%) have worst delivery rates\n")
    f.write("- Top complaint: Late delivery (25.6% of negative reviews)\n")

print("README written!")

README written!
